<div dir="rtl">

# תמלול ראיונות וזיהוי דוברים בעברית — Kaggle

מחברת זו מתמללת קבצי אודיו/וידאו בעברית ומזהה מי מהדוברים אמר כל משפט.

היא מבוססת על [המחברת המקורית](https://github.com/Sourasky-DHLAB/Whisper) של
[הספרייה המרכזית ע"ש סוראסקי](https://cenlib.tau.ac.il/), אוניברסיטת תל אביב (עודד זרחיה),
ונכתבה מחדש ב-2026 כדי לרוץ על **Kaggle** במקום Google Colab, עם מודלים עדכניים:

| רכיב | מה בשימוש | למה |
|---|---|---|
| מנוע תמלול | [faster-whisper](https://github.com/SYSTRAN/faster-whisper) | מהיר פי ~4 מ-`openai-whisper` וצורך פחות זיכרון GPU |
| מודל | [`ivrit-ai/whisper-large-v3-turbo-ct2`](https://huggingface.co/ivrit-ai/whisper-large-v3-turbo-ct2) | Whisper שכוונן במיוחד לעברית — דיוק גבוה משמעותית מהמודל הרגיל |
| זיהוי דוברים | [`pyannote/speaker-diarization-community-1`](https://huggingface.co/pyannote/speaker-diarization-community-1) | מזהה החלפת דובר גם באמצע משפט, ואינו דורש לדעת מראש כמה דוברים יש |
| קלט | ffmpeg | מקבל **כל** פורמט אודיו/וידאו — אין צורך להמיר ל-WAV ידנית |

**השימוש חינמי לחלוטין.** Kaggle מעניק כ-30 שעות GPU בשבוע.

</div>

---

### Before you start — three one-time setup steps

**1. Turn on the GPU.** Right sidebar → **Session options** → **Accelerator** → `GPU T4 x2` or `GPU P100`.

**2. Turn on the internet.** Right sidebar → **Session options** → **Internet** → **On**.
(Kaggle requires a phone-verified account for this. Without it, the models can't download.)

**3. Get a Hugging Face token and accept the model terms.** The diarization model is free but gated:

  - Create a free account at [huggingface.co](https://huggingface.co/join)
  - Visit [pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1) and click **Agree and access repository**
  - Create a **read** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
  - In Kaggle: **Add-ons → Secrets → Add a new secret**, name it exactly `HF_TOKEN`, paste the token, and tick the checkbox to attach it to this notebook

**4. Upload your audio.** Right sidebar → **Upload → New Dataset**, name it `audiofiles`, drop your recordings in.
They will appear under `/kaggle/input/audiofiles/`.


<div dir="rtl">

## 1. בדיקת המעבד הגרפי

הריצו את התא הבא כדי לוודא שהוקצה GPU. אם הפלט ריק או שגוי — חזרו להגדרות והפעילו את ה-Accelerator.

</div>

In [ ]:
!nvidia-smi

<div dir="rtl">

## 2. התקנת ספריות

</div>

We deliberately **do not install or pin `torch`**. Kaggle ships a PyTorch build matched to its own CUDA
driver; overriding it is what broke the original notebook. We only add the two libraries on top.

This cell takes 2-3 minutes.

> **A long red `ERROR: pip's dependency resolver...` block here is expected. Ignore it.**
> Kaggle's image ships hundreds of packages that were already inconsistent with each other, and pip
> audits *all* of them after any install. Some entries even refer to packages that were missing before
> this notebook ran.
>
> The check that matters is the small version table printed at the end of this cell. If those lines
> appear, the install worked — pip would have stopped before reaching them otherwise.

In [ ]:
%pip install -q "faster-whisper>=1.1.0" "pyannote.audio>=4.0.0" "srt>=3.5"

# Show what we actually ended up with, so version problems are visible immediately.
import importlib.metadata as md
import torch
for pkg in ("torch", "faster-whisper", "pyannote.audio", "ctranslate2"):
    try:
        print(f"{pkg:20s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:20s} NOT INSTALLED")
print(f"{'CUDA available':20s} {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"{'GPU':20s} {torch.cuda.get_device_name(0)}")
    # cuDNN is printed because a mismatch between its major version and the one
    # ctranslate2 was built against is the classic way faster-whisper fails, and
    # the error it produces names a library file rather than the real problem.
    # Having the number here means it is already on screen if that ever happens.
    print(f"{'cuDNN':20s} {torch.backends.cudnn.version()}")

<div dir="rtl">

## 3. טוקן Hugging Face

</div>

Reads the `HF_TOKEN` secret you configured in **Add-ons → Secrets**. Nothing is printed —
the token never appears in the notebook output.

In [ ]:
import os

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    # Not on Kaggle, or the secret is not attached to this notebook.
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "No Hugging Face token found.\n"
        "On Kaggle: Add-ons -> Secrets -> add a secret named exactly HF_TOKEN, "
        "then tick the checkbox next to it to attach it to this notebook.\n"
        "Elsewhere: set the HF_TOKEN environment variable."
    )

os.environ["HF_TOKEN"] = HF_TOKEN
print(f"Token loaded ({len(HF_TOKEN)} chars).")

<div dir="rtl">

## 4. הגדרות

זהו התא היחיד שרוב המשתמשים צריכים לשנות.

</div>

| Setting | What it does |
|---|---|
| `INPUT_DIR` | Where your uploaded files live. Leave as-is if your dataset is named `audiofiles`. |
| `LANGUAGE` | `he` for Hebrew, `ar` for Arabic, `en` for English. The ivrit-ai model is Hebrew-only — switch `WHISPER_MODEL` to `large-v3` for other languages. |
| `NUM_SPEAKERS` | Leave `None` to let the model figure it out. Set an integer only if you *know* the count and the automatic result is wrong. |
| `MIN/MAX_SPEAKERS` | A softer hint than `NUM_SPEAKERS` — e.g. `min=2, max=4`. Ignored if `NUM_SPEAKERS` is set. |

In [ ]:
from pathlib import Path

# Everything worth changing lives in this one cell, so you never have to hunt
# through the notebook for a path or a model name.

# --- input / output -------------------------------------------------------
# On Kaggle, /kaggle/input is read-only and holds your uploaded Dataset.
# /kaggle/working is the only writable folder, and it is what the Output panel
# on the right offers for download.
INPUT_DIR  = Path("/kaggle/input/audiofiles")
OUTPUT_DIR = Path("/kaggle/working/transcriptions")

# --- transcription --------------------------------------------------------
# A Whisper that was further trained on Hebrew, so it is far more accurate on
# Hebrew than the standard model. That training degraded its ability to *guess*
# the language, which is why LANGUAGE is always passed explicitly rather than
# left to autodetect. For any other language, switch to "large-v3".
WHISPER_MODEL = "ivrit-ai/whisper-large-v3-turbo-ct2"
LANGUAGE      = "he"

# --- speaker diarization --------------------------------------------------
DIARIZATION_MODEL = "pyannote/speaker-diarization-community-1"
# Leave all three as None and the model works out how many speakers there are by
# itself, which it is usually good at. Override only if that answer comes out
# wrong: NUM_SPEAKERS forces an exact count, while MIN/MAX give it a range to
# stay inside, which is the gentler hint.
NUM_SPEAKERS = None
MIN_SPEAKERS = None
MAX_SPEAKERS = None

# --- output ---------------------------------------------------------------
# The model labels people SPEAKER_00, SPEAKER_01 ... in the order it happens to
# find them, so which label is which person is arbitrary. Run once, read the
# preview at the bottom, then come back and put real names here.
SPEAKER_NAMES = {}     # e.g. {"SPEAKER_00": "מראיין", "SPEAKER_01": "מרואיינת"}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output folder: {OUTPUT_DIR}")

<div dir="rtl">

## 5. איתור קבצי הקלט

התא הבא סורק את תיקיית הקלט ומציג את הקבצים שיתומללו.

</div>

In [ ]:
# Video formats are included on purpose: the audio is pulled out of them in the
# next cell, so an .mp4 straight off a phone works with no preparation. This set
# exists only to skip the stray .csv, .txt or .DS_Store that tends to ride along
# in an uploaded dataset.
MEDIA_EXT = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".opus", ".aac", ".wma",
             ".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"}

# Both checks below stop the notebook immediately with an explanation. The
# alternative is a confusing empty result several cells later, by which point the
# real cause is hard to see.
if not INPUT_DIR.exists():
    # /kaggle/input is read-only. You cannot create this folder from code - it is
    # filled in by Kaggle when you attach a Dataset. So a missing folder means
    # either no Dataset is attached, or yours is attached under a different name
    # (Kaggle turns "My Audio" into "my-audio"). Listing what is actually mounted
    # turns a dead end into an answer.
    kaggle_input = Path("/kaggle/input")
    attached = sorted(p.name for p in kaggle_input.iterdir()) if kaggle_input.exists() else []
    if attached:
        hint = (f"Datasets attached right now: {', '.join(attached)}.\n"
                "If yours is in that list, change INPUT_DIR above to match it.\n")
    else:
        hint = "No datasets are attached to this notebook yet.\n"
    raise FileNotFoundError(
        f"{INPUT_DIR} does not exist.\n" + hint +
        "To attach one: right sidebar -> Add Input -> Upload -> New Dataset, named 'audiofiles'.\n"
        "Note: /kaggle/input is read-only, so this folder cannot be created by code."
    )

# rglob searches subfolders too, because uploading a zip keeps its folder
# structure and the files often end up a level down rather than at the top.
media_files = sorted(p for p in INPUT_DIR.rglob("*") if p.suffix.lower() in MEDIA_EXT)

if not media_files:
    raise FileNotFoundError(f"No audio or video files found under {INPUT_DIR}")

for p in media_files:
    print(f"  {p.name}  ({p.stat().st_size / 1e6:.1f} MB)")
print(f"\n{len(media_files)} file(s) to process.")

<div dir="rtl">

## 6. הכנת האודיו

</div>

The original notebook required you to manually convert everything to mono WAV first.
This cell does it for you with ffmpeg — **any** audio or video format works, including MP4 straight
from a phone. Output is 16 kHz mono WAV, which is exactly what both models want.

In [ ]:
import subprocess, shutil, tempfile

if shutil.which("ffmpeg") is None:
    raise RuntimeError("ffmpeg not found on this machine.")

# A scratch folder, deliberately not inside /kaggle/working: these converted
# files are throwaway, and putting them in the output folder would bury your
# actual transcripts among large WAVs in the download panel.
PREPARED_DIR = Path(tempfile.mkdtemp(prefix="prepared_audio_"))

def prepare_audio(src: Path) -> Path:
    """Convert any media file to the one audio format both models want.

    Whisper and pyannote each expect 16 kHz mono and will convert anything else
    themselves. Doing it once here buys two things: you never have to convert
    files by hand (the original notebook demanded a mono WAV up front), and both
    models then analyse byte-identical audio, so the timestamps they produce
    refer to exactly the same thing when we match them up later.
    """
    dst = PREPARED_DIR / (src.stem + ".wav")
    cmd = ["ffmpeg", "-y", "-loglevel", "error",
           "-i", str(src),
           "-vn",              # ignore any video stream; decoding pictures is wasted work
           "-ac", "1",         # mono: one channel is what the models read
           "-ar", "16000",     # 16 kHz: speech models are trained at this rate
           "-c:a", "pcm_s16le",  # uncompressed, so no second round of lossy encoding
           str(dst)]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"ffmpeg failed on {src.name}:\n{proc.stderr}")
    return dst

prepared = {}
for p in media_files:
    prepared[p] = prepare_audio(p)
    dur = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", str(prepared[p])],
        capture_output=True, text=True).stdout.strip()
    print(f"  {p.name} -> {float(dur)/60:.1f} min")

print(f"\nPrepared {len(prepared)} file(s).")

<div dir="rtl">

## 7. טעינת המודלים

הורדת המודלים בפעם הראשונה אורכת מספר דקות (כ-1.6 ג'יגה-בייט).

</div>

In [ ]:
import torch
from faster_whisper import WhisperModel
from pyannote.audio import Pipeline

# compute_type is the number format the model calculates in. float16 uses half
# the memory of the default and runs faster on a GPU, with no accuracy loss worth
# worrying about for speech. CPUs have no useful float16 support, so there we
# fall back to int8, which is cruder but at least runs.
if torch.cuda.is_available():
    device, compute_type = "cuda", "float16"
else:
    device, compute_type = "cpu", "int8"
    print("WARNING: no GPU detected. This will be extremely slow. "
          "Enable the accelerator in Session options.")

print(f"Device: {device} ({compute_type})")

print("Loading transcription model ...")
whisper_model = WhisperModel(WHISPER_MODEL, device=device, compute_type=compute_type)

print("Loading diarization pipeline ...")
diarization_pipeline = Pipeline.from_pretrained(DIARIZATION_MODEL, token=HF_TOKEN)
if diarization_pipeline is None:
    raise RuntimeError(
        f"Could not load {DIARIZATION_MODEL}.\n"
        "Most likely you have not accepted the model terms. Open "
        f"https://huggingface.co/{DIARIZATION_MODEL} while logged in and click "
        "'Agree and access repository', then re-run this cell."
    )
diarization_pipeline.to(torch.device(device))

print("Both models ready.")

<div dir="rtl">

## 8. פונקציות עזר

התא הבא מגדיר את הלוגיקה שמשייכת כל מילה לדובר. אין צורך לשנות בו דבר.

</div>

**How speaker assignment works.** The original notebook gave one speaker per Whisper segment, so a
speaker change mid-sentence was impossible to represent. Here we ask Whisper for **word-level
timestamps**, then match every individual word to the diarization turn it overlaps most. Consecutive
words with the same speaker are then merged back into readable blocks.

In [ ]:
def format_timestamp(seconds: float, srt: bool = False) -> str:
    """HH:MM:SS, or HH:MM:SS,mmm for SRT.

    Work in integer milliseconds throughout: float truncation otherwise renders
    2.4s as '00:00:02,399', which is off by a millisecond on every cue.
    """
    seconds = max(0.0, seconds)
    if srt:
        total_ms = int(round(seconds * 1000))
        h, rem = divmod(total_ms, 3_600_000)
        m, rem = divmod(rem, 60_000)
        s, ms = divmod(rem, 1000)
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"
    total = int(seconds)
    return f"{total // 3600:02d}:{(total % 3600) // 60:02d}:{total % 60:02d}"


def assign_speaker_to_words(words, turns):
    """For each word, pick the diarization turn with the largest time overlap.

    words: sequence of objects with .start/.end/.word
    turns: list of (start, end, speaker), sorted by start
    Returns a list of speaker labels, one per word.
    """
    labels, last = [], None
    for w in words:
        best_spk, best_overlap, best_len = None, 0.0, float("inf")
        for start, end, spk in turns:
            if end <= w.start:
                continue
            if start >= w.end:
                break          # turns are sorted; nothing further can overlap
            overlap = min(end, w.end) - max(start, w.start)
            turn_len = end - start
            # Prefer the larger overlap; on a tie prefer the tighter turn, which
            # is the more specific match when turns overlap each other.
            if overlap > best_overlap or (
                overlap == best_overlap and overlap > 0 and turn_len < best_len
            ):
                best_overlap, best_spk, best_len = overlap, spk, turn_len
        # A word landing in a diarization gap inherits the previous speaker.
        if best_spk is None:
            best_spk = last
        last = best_spk
        labels.append(best_spk)
    return labels


def group_into_blocks(words, labels):
    """Merge runs of consecutive same-speaker words into blocks."""
    blocks = []
    for w, spk in zip(words, labels):
        if blocks and blocks[-1]["speaker"] == spk:
            blocks[-1]["end"] = w.end
            blocks[-1]["text"] += w.word
        else:
            blocks.append({"speaker": spk, "start": w.start,
                           "end": w.end, "text": w.word})
    for b in blocks:
        b["text"] = b["text"].strip()
    return [b for b in blocks if b["text"]]


def display_name(raw_label):
    if raw_label is None:
        return "דובר לא ידוע"
    if raw_label in SPEAKER_NAMES:
        return SPEAKER_NAMES[raw_label]
    # SPEAKER_00 -> "דובר 1"
    digits = "".join(ch for ch in raw_label if ch.isdigit())
    return f"דובר {int(digits) + 1}" if digits else raw_label

print("Helpers defined.")

<div dir="rtl">

## 9. תמלול וזיהוי דוברים

זהו התא הכבד. משך הריצה תלוי באורך הקובץ — כשעה של אודיו לוקחת בערך 5-10 דקות על T4.

</div>

In [ ]:
import time, gc

results = {}

for original, wav in prepared.items():
    print(f"\n{'=' * 70}\n{original.name}\n{'=' * 70}")
    t0 = time.time()

    # ---- 1. transcribe, with word-level timestamps ------------------------
    print("  Transcribing ...")
    segments_gen, info = whisper_model.transcribe(
        str(wav),
        language=LANGUAGE,
        word_timestamps=True,
        vad_filter=True,                       # skip long silences
        vad_parameters={"min_silence_duration_ms": 500},
    )
    segments = list(segments_gen)              # generator -> list (this is where work happens)
    words = [w for seg in segments for w in (seg.words or [])]
    print(f"    {len(segments)} segments, {len(words)} words "
          f"({time.time() - t0:.0f}s)")

    if not words:
        print("    No speech detected - skipping.")
        continue

    # ---- 2. diarize -------------------------------------------------------
    print("  Identifying speakers ...")
    t1 = time.time()
    kwargs = {}
    if NUM_SPEAKERS:
        kwargs["num_speakers"] = NUM_SPEAKERS
    else:
        if MIN_SPEAKERS:
            kwargs["min_speakers"] = MIN_SPEAKERS
        if MAX_SPEAKERS:
            kwargs["max_speakers"] = MAX_SPEAKERS

    dia = diarization_pipeline(str(wav), **kwargs)

    # community-1 exposes an "exclusive" view: non-overlapping turns, which is
    # what we want when reconciling against transcription timestamps.
    annotation = getattr(dia, "exclusive_speaker_diarization", None)
    if annotation is None:
        annotation = getattr(dia, "speaker_diarization", dia)

    turns = sorted(
        ((turn.start, turn.end, spk) for turn, _, spk in annotation.itertracks(yield_label=True)),
        key=lambda t: t[0],
    )
    speakers_found = sorted({spk for _, _, spk in turns})
    print(f"    {len(turns)} turns, {len(speakers_found)} speaker(s): "
          f"{', '.join(speakers_found)} ({time.time() - t1:.0f}s)")

    # ---- 3. merge ---------------------------------------------------------
    labels = assign_speaker_to_words(words, turns)
    blocks = group_into_blocks(words, labels)
    results[original] = {"blocks": blocks, "segments": segments, "turns": turns}

    print(f"  Done in {time.time() - t0:.0f}s -> {len(blocks)} speaker blocks")

    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

print(f"\n\nProcessed {len(results)} file(s).")

<div dir="rtl">

## 10. שמירת התמלילים

הקבצים נשמרים תחת `/kaggle/working/transcriptions/`. להורדה: לחצו על **Output** בסרגל הימני.

</div>

In [ ]:
# U+200F, the right-to-left mark. It is invisible, but without it an editor that
# defaults to left-to-right will mangle a line that mixes Hebrew with digits or
# Latin letters - which every line here does, because of the timestamps.
RLM = "\u200f"

def write_txt(path, blocks):
    with open(path, "w", encoding="utf-8") as f:
        for b in blocks:
            f.write(f"\n{RLM}[{format_timestamp(b['start'])}] "
                    f"{display_name(b['speaker'])}:\n")
            f.write(f"{RLM}{b['text']}\n")


def write_srt(path, segments, turns):
    """Subtitle file: one cue per Whisper segment, labelled with its speaker.

    Cues follow Whisper's own segments rather than the regrouped speaker blocks
    of the txt file. A block can run for a minute of one person talking, which
    reads fine on a page but is far too long to sit on screen as a subtitle.
    """
    with open(path, "w", encoding="utf-8") as f:
        idx = 0
        for seg in segments:
            if not (seg.text or "").strip():
                continue
            idx += 1
            spk = assign_speaker_to_words([seg], turns)[0]
            f.write(f"{idx}\n")
            f.write(f"{format_timestamp(seg.start, srt=True)} --> "
                    f"{format_timestamp(seg.end, srt=True)}\n")
            f.write(f"{RLM}{display_name(spk)}: {seg.text.strip()}\n\n")


written = []
for original, data in results.items():
    txt_path = OUTPUT_DIR / f"{original.stem}.txt"
    write_txt(txt_path, data["blocks"])
    written.append(txt_path)

    srt_path = OUTPUT_DIR / f"{original.stem}.srt"
    write_srt(srt_path, data["segments"], data["turns"])
    written.append(srt_path)

for p in written:
    print(f"  {p}  ({p.stat().st_size / 1024:.1f} KB)")
print(f"\nWrote {len(written)} file(s).")

<div dir="rtl">

## 11. תצוגה מקדימה

התא הבא מציג את תחילת התמליל, מיושר לימין.

</div>

In [ ]:
from IPython.display import HTML, display
import html as html_lib

PREVIEW_BLOCKS = 25

for original, data in results.items():
    rows = []
    for b in data["blocks"][:PREVIEW_BLOCKS]:
        rows.append(
            f'<div style="margin-bottom:0.9em">'
            f'<b>{html_lib.escape(display_name(b["speaker"]))}</b> '
            f'<span style="color:#888;font-size:0.85em">'
            f'[{format_timestamp(b["start"])}]</span><br>'
            f'{html_lib.escape(b["text"])}</div>'
        )
    more = ""
    if len(data["blocks"]) > PREVIEW_BLOCKS:
        more = (f'<div style="color:#888">... ועוד '
                f'{len(data["blocks"]) - PREVIEW_BLOCKS} קטעים</div>')
    display(HTML(
        f'<div dir="rtl" style="text-align:right;font-size:1.05em;'
        f'line-height:1.6;font-family:Arial,sans-serif">'
        f'<h3>{html_lib.escape(original.name)}</h3>'
        f'{"".join(rows)}{more}</div>'
    ))

---

<div dir="rtl">

## פתרון תקלות

</div>

| Symptom | Cause and fix |
|---|---|
| `Could not load pyannote/...` or a 401 error | You have not accepted the model terms, or `HF_TOKEN` is wrong. Open the model page while logged in, click **Agree and access repository**, and check the secret is *attached* (ticked) in Add-ons → Secrets. |
| `Could not load library libcudnn_ops.so` | cuDNN mismatch in the Kaggle image. Set `compute_type = "int8_float16"` in the model-loading cell. |
| Everything labelled as one speaker | The recording may be mono-mixed with very similar voices. Try setting `NUM_SPEAKERS` explicitly. |
| Too many speakers detected | Background noise or music. Set `MAX_SPEAKERS`, or clean the audio first. |
| Session dies partway through | Kaggle caps sessions at 12h and RAM at ~30 GB. Process fewer files per run. |
| Hebrew displays backwards in Notepad | Open it in an RTL-aware editor (Word, VS Code, Google Docs). The output already carries invisible right-to-left marks, which most viewers honour; Notepad does not. |
| Transcription is poor quality | Confirm `LANGUAGE = "he"`. The ivrit-ai model is Hebrew-only — for other languages switch `WHISPER_MODEL` to `"large-v3"`. |

<div dir="rtl">

### קרדיטים

מבוסס על המחברת המקורית של [הספרייה המרכזית ע"ש סוראסקי](https://cenlib.tau.ac.il/), אוניברסיטת תל אביב.
מודל התמלול בעברית: [ivrit.ai](https://www.ivrit.ai/). זיהוי דוברים: [pyannote.audio](https://github.com/pyannote/pyannote-audio).

</div>